In [ ]:
import os, re, subprocess, sys, time, urllib.request, pathlib, shutil
from google.colab import drive
repo = "https://github.com/Prajwalx69/KYLP-CLIPS"
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
work = "/content/klyp_public"
shutil.rmtree(work, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", repo + ".git", work], check=True)
hits = list(pathlib.Path(work).rglob("klyp_server.py"))
if not hits:
    raise RuntimeError("klyp_server.py not found in repository")
app_dir = hits[0].parent
req = app_dir / "requirements-klyp.txt"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)
drive.mount("/content/drive")
port = 7861
server_log = open("/content/klyp_server.log", "w")
server = subprocess.Popen([sys.executable, "klyp_server.py", "--port", str(port)], cwd=str(app_dir), stdout=server_log, stderr=subprocess.STDOUT)
time.sleep(4)
cf = "/usr/local/bin/cloudflared"
if not os.path.exists(cf):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cf)
    os.chmod(cf, 0o755)
tunnel_log = open("/content/klyp_tunnel.log", "w")
tunnel = subprocess.Popen([cf, "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"], stdout=tunnel_log, stderr=subprocess.STDOUT)
link = None
for _ in range(60):
    tunnel_log.flush()
    try:
        content = pathlib.Path("/content/klyp_tunnel.log").read_text(errors="ignore")
    except Exception:
        content = ""
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", content)
    if match:
        link = match.group(0)
        break
    time.sleep(1)
if link:
    print("KLYP URL:", link)
else:
    print("Tunnel link not found. Check /content/klyp_tunnel.log")
try:
    while server.poll() is None:
        time.sleep(20)
except KeyboardInterrupt:
    server.terminate()
    tunnel.terminate()
